# PCAのインタラクティブデモ / Interactive PCA Demo

**主成分分析を「軸を見つける → 射影する → 圧縮・復元する」の順に体験します。**  
**Explore PCA: find axes → project data → compress and reconstruct.**

Google Colabで **「ランタイム → すべてのセルを実行」** を選び、各セクションのスライダーを操作してください。GPUは不要です。  
In Google Colab, choose **Runtime → Run all**, then use the sliders. No GPU is required.

| デモ / Demo | 操作と観察 / Controls and observations |
|---|---|
| 1. 主成分と射影 / Principal axes and projection | 点群の方向・細長さ・射影軸を変更し、分散と復元誤差を観察 / Change cloud orientation, elongation and projection angle; inspect variance and error |
| 2. 単位と標準化 / Units and standardization | 一方の変数の尺度を変え、標準化の有無を比較 / Rescale one feature and compare PCA with and without standardization |
| 3. 画像の圧縮・復元 / Image compression and reconstruction | 残す主成分数を変え、未学習画像の復元と寄与率を観察 / Vary retained components and inspect held-out image reconstruction |

**学習目標 / Learning goals**
- PCAは変数を選ぶのではなく、線形結合による新しい軸を作ります。 / PCA constructs new axes as linear combinations of features.
- 固定した次元数では、PCAは二乗復元誤差を最小化します。 / For a fixed dimension, PCA minimizes squared reconstruction error.
- 寄与率は「分散の保持率」であり、「意味や識別情報の保持率」ではありません。 / Explained variance is not the fraction of semantic or discriminative information preserved.


In [ ]:
# 依存関係と表示の設定 / Dependencies and display setup
import importlib.util
import subprocess
import sys

# 不足分だけ導入します。Colabでは日本語フォント用の追加が主です。
# Install missing packages only; Colab mainly needs Japanese font support.
packages = {"numpy": "numpy", "matplotlib": "matplotlib", "sklearn": "scikit-learn",
            "ipywidgets": "ipywidgets", "japanize_matplotlib": "japanize-matplotlib"}
for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import ipywidgets as widgets
from IPython.display import display, HTML
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # 通常のJupyterでも使用可能 / Also works in standard Jupyter.

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN = "#1769aa", "#e67e22", "#15946e"
STYLE = {"description_width": "210px"}
LAYOUT = widgets.Layout(width="620px")

def slider(label, value, minimum, maximum, step=1, floating=False):
    cls = widgets.FloatSlider if floating else widgets.IntSlider
    return cls(description=label, value=value, min=minimum, max=maximum, step=step,
               continuous_update=False, style=STYLE, layout=LAYOUT)

def cloud(angle=35, ratio=0.3, n=160, seed=7):
    """回転したガウス点群 / Rotated Gaussian cloud."""
    rng = np.random.default_rng(seed)
    a = np.deg2rad(angle)
    rotation = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    return (rng.normal(size=(n, 2)) * [2.5, 2.5 * ratio]) @ rotation.T

print("準備完了 / Ready — GPU不要 / CPU only")


## 1. 主成分の方向と射影 / Principal axes and projection

**操作 / Try this**
1. 「PC1を使う」をオンにし、点群の方向を変えます。PC1が点群に追随することを確認します。 / Keep “Use PC1” on and rotate the cloud.
2. オフにして、射影角度を手動で変えます。射影後の分散が最大になる方向と、復元誤差が最小になる方向を探します。 / Turn it off; find the angle maximizing projected variance and minimizing reconstruction error.
3. 細長さを1に近づけます。分散が同程度になると、代表的な方向が決まりにくくなります。有限標本では完全な円にはなりません。 / Move the aspect ratio toward 1; an isotropic population has no unique principal direction, while finite samples retain small differences.

中心化したデータを $X_c=X-\mu$、単位射影軸を $u$ とすると、  
For centered data and a unit projection axis:

$$z=X_cu,\qquad \hat X=zu^\top+\mu$$

$$\mathrm{MSE}=\frac{1}{nd}\sum_{i=1}^n\|x_i-\hat x_i\|^2$$

分散曲線は母分散形式（ddof=0）で描き、復元誤差との恒等式をそろえています。主成分軸の符号には任意性があります。  
Variance curves use ddof=0 for consistency with reconstruction MSE. Principal axis signs are arbitrary.


In [ ]:
# デモ1の描画 / Demo 1 rendering

def demo_projection(data_angle=35, ratio=0.3, projection_angle=35, use_pc1=True, residuals=True):
    X = cloud(data_angle, ratio)
    pca = PCA(n_components=2, svd_solver="full").fit(X)
    mean = pca.mean_
    C = X - mean
    pc1, pc2 = pca.components_
    theta = np.arctan2(pc1[1], pc1[0]) if use_pc1 else np.deg2rad(projection_angle)
    u = np.array([np.cos(theta), np.sin(theta)])
    z = C @ u
    reconstructed = z[:, None] * u + mean
    mse = np.mean((X - reconstructed) ** 2)
    angles = np.linspace(0, 180, 361)
    directions = np.column_stack([np.cos(np.deg2rad(angles)), np.sin(np.deg2rad(angles))])
    variances = np.var(C @ directions.T, axis=0)
    errors = (np.sum(C ** 2, axis=1).mean() - variances) / 2
    current = np.rad2deg(theta) % 180
    lim = max(7.0, np.max(np.abs(X)) * 1.12)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    ax = axes[0]
    ax.scatter(*X.T, s=16, alpha=.5, color=BLUE, label="元データ / Data")
    if residuals:
        for x, r in zip(X, reconstructed):
            ax.plot([x[0], r[0]], [x[1], r[1]], color="#aaaaaa", alpha=.35, lw=.6)
    ax.scatter(*reconstructed.T, s=10, color=ORANGE, label="1次元復元 / Rank-1 reconstruction")
    for v, name, color in [(pc1, "PC1", GREEN), (pc2, "PC2", "#8e44ad")]:
        ax.annotate("", xy=mean + 3.7*v, xytext=mean,
                    arrowprops=dict(arrowstyle="->", color=color, lw=2))
        ax.text(*(mean + 3.9*v), name, color=color)
    line = mean + np.array([-lim, lim])[:, None] * u
    ax.plot(*line.T, color=ORANGE, lw=1, label="射影軸 / Projection axis")
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel="特徴1 / Feature 1", ylabel="特徴2 / Feature 2",
           title="主成分と射影 / Principal axes and projection")
    ax.set_aspect("equal"); ax.legend(fontsize=8, loc="lower left"); ax.grid(alpha=.15)

    axes[1].hist(z, bins=22, color=ORANGE, alpha=.8)
    axes[1].set(title="圧縮後は1つの数値 / One coordinate per sample", xlabel="潜在座標 z / Projected coordinate", ylabel="件数 / Count")
    axes[1].text(.04, .96, f"保持分散 / Retained variance: {np.var(z)/np.var(C,axis=0).sum():.1%}\n"
                 f"復元MSE / Reconstruction MSE: {mse:.4f}", transform=axes[1].transAxes,
                 va="top", bbox=dict(facecolor="white", alpha=.85, edgecolor="none"))

    ax = axes[2]
    ax.plot(angles, variances, color=GREEN, label="射影後の分散 / Projected variance")
    ax.axvline(current, color=ORANGE, ls="--", label="選択軸 / Selected axis")
    ax.scatter([current], [np.var(z)], color=ORANGE, zorder=5)
    ax.set(xlabel="射影角度 / Projection angle (degrees)", ylabel="分散 / Variance", xlim=(0,180),
           title="PC1は分散最大・MSE最小 / PC1: max variance, min MSE")
    right = ax.twinx()
    right.plot(angles, errors, color=BLUE, ls=":", label="復元MSE / Reconstruction MSE")
    right.set_ylabel("復元MSE / Reconstruction MSE", color=BLUE)
    handles, labels = ax.get_legend_handles_labels()
    h2, l2 = right.get_legend_handles_labels()
    ax.legend(handles+h2, labels+l2, fontsize=7, loc="lower center")
    plt.show()

controls1 = {
    "data_angle": slider("点群の角度 / Cloud angle", 35, 0, 175, 5),
    "ratio": slider("短軸/長軸 / Aspect ratio", .3, .05, 1, .05, True),
    "projection_angle": slider("射影角度 / Projection angle", 35, 0, 180, 1),
    "use_pc1": widgets.Checkbox(description="PC1を使う / Use PC1 (manual angle ignored)", value=True, indent=False),
    "residuals": widgets.Checkbox(description="復元誤差を線で表示 / Show residual lines", value=True, indent=False)
}
projection_output = widgets.interactive_output(demo_projection, controls1)
display(widgets.VBox(list(controls1.values())), projection_output)


## 2. 単位の影響と標準化 / Units and standardization

同じ対象でも、単位を変えると分散の大きさが変わり、PCAの結果が変わります。ここでは特徴2だけを倍率で拡大します。  
Changing measurement units changes variance and can change PCA. Here only Feature 2 is rescaled.

- **標準化なし / Without standardization:** 入力の単位で復元誤差を最小化します。 / Minimize reconstruction error in input units.
- **標準化あり / With standardization:** 各変数を平均0・分散1に変換した空間でPCAを計算し、図では元の単位へ戻します。 / Fit PCA after scaling each feature to zero mean and unit variance; map reconstructions back to input units for display.

標準化が常に正解とは限りません。単位の違いを除きたいのか、絶対的なばらつきを重視するのかで選びます。左右の寄与率は**異なる尺度の空間**で計算されるため、直接の優劣比較はできません。  
Standardization is a modeling choice, not always an improvement. Explained-variance ratios in the two panels use different metrics and are not directly comparable.


In [ ]:
# デモ2の描画 / Demo 2 rendering

def demo_scaling(scale=4.0):
    X = cloud(35, .38, seed=21) * [1, scale]
    scaler = StandardScaler().fit(X)
    S = scaler.transform(X)
    raw = PCA(1, svd_solver="full").fit(X)
    std = PCA(1, svd_solver="full").fit(S)
    raw_rec = raw.inverse_transform(raw.transform(X))
    std_rec = scaler.inverse_transform(std.inverse_transform(std.transform(S)))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    for ax, reconstructed, model, title, metric in [
        (axes[0], raw_rec, raw, "標準化なし / Without standardization", "入力単位 / Input units"),
        (axes[1], std_rec, std, "標準化あり / With standardization", "標準化空間 / Standardized space")]:
        ax.scatter(*X.T, s=17, color=BLUE, alpha=.55, label="元データ / Data")
        ax.scatter(*reconstructed.T, s=12, color=ORANGE, alpha=.8, label="1次元復元 / Rank-1 reconstruction")
        ax.set(title=title, xlabel="特徴1 / Feature 1", ylabel=f"特徴2 × {scale:g} / Feature 2 × {scale:g}")
        ax.set_xlim(X[:,0].min()-1, X[:,0].max()+1)
        ax.set_ylim(X[:,1].min()-scale, X[:,1].max()+scale)
        ax.grid(alpha=.2); ax.legend(fontsize=8, loc="lower right")
        ax.text(.03, .97, f"PC1寄与率 / Explained variance: {model.explained_variance_ratio_[0]:.1%}\n"
                f"評価尺度 / Metric: {metric}", transform=ax.transAxes, va="top", fontsize=9,
                bbox=dict(facecolor="white", alpha=.85, edgecolor="none"))
    plt.show()

scale_control = slider("特徴2の倍率 / Feature 2 scale", 4., .25, 10., .25, True)
scaling_output = widgets.interactive_output(demo_scaling, {"scale": scale_control})
display(scale_control, scaling_output)


## 3. 主成分数と画像の復元 / Components and image reconstruction

scikit-learn付属の8×8画素の数字画像を使います。追加のデータダウンロードは不要です。**PCAの学習には訓練画像のみを使い、表示はテスト画像**です。数字ラベルは表示とデータ分割にのみ使い、PCAの学習には使いません。  
Use the bundled 8×8 digit images; no dataset download is required. **Fit PCA on training images only and display held-out test images.** Labels are used only for display and stratified splitting, not PCA fitting.

**操作 / Try this**
- 主成分数を1 → 5 → 15 → 30 → 64と増やします。 / Increase retained components from 1 to 5, 15, 30, and 64.
- 寄与率と復元誤差を比較します。高い寄与率でも、小さな特徴が失われる場合があります。 / Compare explained variance and reconstruction error; subtle features may be lost even with high explained variance.
- 64成分では元の64次元を保持するため、丸め誤差を除いて元画像に戻ります。 / All 64 components reconstruct the input up to numerical precision.

$$z=(x-\mu)U_k,\qquad \hat{x}=zU_k^\top+\mu$$

表示する「係数数比率 k/64」は画像ごとの座標数の比率です。実際のファイル圧縮率ではなく、平均画像・主成分行列の保存コストは含みません。  
The coefficient ratio k/64 counts per-image coordinates, not file size; it excludes the shared mean and PCA basis.


In [ ]:
# 一度だけ学習してキャッシュ / Fit once and cache transformations
_digits = load_digits()
_train_idx, _test_idx = train_test_split(
    np.arange(len(_digits.data)), test_size=.3, random_state=42, stratify=_digits.target)
X_train = _digits.data[_train_idx] / 16.0
X_test = _digits.data[_test_idx] / 16.0
y_test = _digits.target[_test_idx]
digit_pca = PCA(n_components=64, svd_solver="full").fit(X_train)
test_scores = digit_pca.transform(X_test)
cumulative_variance = np.cumsum(digit_pca.explained_variance_ratio_)

# 全テスト画像のMSEも事前計算 / Precompute held-out MSE for every k
all_test_mse = []
for count in range(1, 65):
    restored = test_scores[:, :count] @ digit_pca.components_[:count] + digit_pca.mean_
    all_test_mse.append(float(np.mean((X_test-restored)**2)))
all_test_mse = np.array(all_test_mse)
print(f"訓練 / Train: {len(X_train)} | テスト / Test: {len(X_test)} | 元次元 / Original dimension: 64")

def demo_digits(k=15, page=0):
    ids = np.arange(page*8, page*8+8)
    originals = X_test[ids]
    restored = test_scores[ids, :k] @ digit_pca.components_[:k] + digit_pca.mean_
    errors = np.abs(originals-restored)
    fig, axes = plt.subplots(3, 8, figsize=(14, 5), constrained_layout=True)
    for col, idx in enumerate(ids):
        axes[0,col].imshow(originals[col].reshape(8,8), cmap="gray", vmin=0, vmax=1)
        axes[0,col].set_title(f"数字 / Digit {y_test[idx]}", fontsize=9)
        axes[1,col].imshow(restored[col].reshape(8,8), cmap="gray", vmin=0, vmax=1)
        axes[2,col].imshow(errors[col].reshape(8,8), cmap="magma", vmin=0, vmax=1)
        for row in range(3):
            axes[row,col].set_xticks([]); axes[row,col].set_yticks([])
    for row, label in enumerate(["元画像\nOriginal", "復元\nReconstructed", "絶対誤差\nAbsolute error"]):
        axes[row,0].set_ylabel(label)
    fig.suptitle(f"保持成分 / Components: {k}/64  |  訓練寄与率 / Train variance: {cumulative_variance[k-1]:.1%}  |  "
                 f"全テストMSE / Test MSE: {all_test_mse[k-1]:.5f}", fontsize=11)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.1), constrained_layout=True)
    axes[0].plot(np.arange(1,65), cumulative_variance, color=GREEN)
    axes[0].axvline(k, color=ORANGE, ls="--")
    axes[0].scatter([k], [cumulative_variance[k-1]], color=ORANGE)
    axes[0].set(xlabel="成分数 k / Components", ylabel="累積寄与率 / Cumulative variance", ylim=(0,1.03),
                title="訓練データの分散保持 / Training variance retained")
    axes[1].plot(np.arange(1,65), all_test_mse, color=BLUE)
    axes[1].axvline(k, color=ORANGE, ls="--")
    axes[1].scatter([k], [all_test_mse[k-1]], color=ORANGE)
    axes[1].set(xlabel="成分数 k / Components", ylabel="復元MSE / Reconstruction MSE",
                title="未学習画像の復元誤差 / Held-out reconstruction error")
    for ax in axes: ax.grid(alpha=.2)
    plt.show()
    print(f"係数数比率 / Coefficient ratio: {k}/64 = {k/64:.1%}（共有基底のコストを除く / excludes shared basis）")
    print("復元値は誤差計算前にクリップしていません。表示範囲だけを0〜1に固定しています。 / Reconstructions are not clipped before MSE; display limits are fixed to [0, 1].")

controls3 = {"k": slider("保持成分数 / Components k", 15, 1, 64),
             "page": slider("表示ページ / Sample page", 0, 0, len(X_test)//8-1)}
digits_output = widgets.interactive_output(demo_digits, controls3)
display(widgets.VBox(list(controls3.values())), digits_output)


## 確認と講義での問い / Checks and discussion

1. なぜ射影後の分散を最大にすると、二乗復元誤差が最小になりますか。 / Why does maximizing projected variance minimize squared reconstruction error?
2. 変数の単位を変えるだけで、PC1が変わるのはなぜですか。 / Why can changing units change PC1?
3. 寄与率95%は、分類に必要な情報を95%保持するという意味ですか。 / Does 95% explained variance mean retaining 95% of the information needed for classification?
4. 線形AEとの共通点は何ですか。 / How is this related to a linear autoencoder?

**線形AEとの関係 / Connection to linear AE**  
同じ中心化データ・潜在次元・二乗復元誤差を用い、追加の正則化がない線形AEを大域最適解まで学習すると、PCAと同じ最適な主部分空間を学びます。ただし潜在座標の軸・尺度・順序は一致するとは限りません。  
With centered data, the same bottleneck dimension, squared reconstruction loss, no extra regularization, and a global optimum, a linear AE learns an optimal PCA principal subspace. Its latent axes, scales and ordering need not match PCA.

**操作できないとき / Troubleshooting**  
ランタイムに接続し、準備セルから順に再実行してください。スライダーはマウスを離したときに更新されます。保存済みの図だけではウィジェットは操作できません。  
Connect a runtime and rerun from the setup cell. Sliders update on release. Saved static figures alone are not interactive.

**参照 / References**
- [scikit-learn: PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [scikit-learn: decomposition](https://scikit-learn.org/stable/modules/decomposition.html)
- [Google Colab: advanced outputs and widgets](https://colab.research.google.com/notebooks/snippets/advanced_outputs.ipynb)
